# CardioIA — Fase 4 | IR ALÉM 1: Ética e Governança em Visão Computacional

Este notebook analisa **vieses, limitações e fairness** do modelo de classificação de raios-X de tórax desenvolvido na Parte 2, propondo práticas de mitigação responsáveis.

> ⚙️ **Como executar:** Google Colab → `Ambiente de execução > Alterar tipo de ambiente > GPU (T4)`  
> Tempo estimado (inclui re-treino): ~18 min no total.

> ⚠️ **Aviso:** Este é um protótipo educacional. Os resultados **não devem** ser usados em decisões clínicas reais sem validação prospectiva e aprovação regulatória.

## 1. Setup — imports e constantes

In [ ]:
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from PIL import Image
from sklearn.calibration import calibration_curve
from sklearn.metrics import (
    auc,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

IMG_SIZE = (150, 150)
BATCH_SIZE = 32
VAL_SPLIT = 0.20
EPOCHS = 15

print("TensorFlow:", tf.__version__)
print("GPU disponível:", tf.config.list_physical_devices("GPU"))

## 2. Pipeline de dados (reprodução fiel do notebook 02)

Reproduzimos aqui as mesmas etapas do `02_cnn_transfer_learning.ipynb` para garantir que o modelo treinado neste notebook receba exatamente os mesmos dados.

In [ ]:
import kagglehub

dataset_path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")
DATA_DIR = Path(dataset_path) / "chest_xray"

train_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train", validation_split=VAL_SPLIT, subset="training", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="binary",
)
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "train", validation_split=VAL_SPLIT, subset="validation", seed=SEED,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="binary",
)
# shuffle=False é crítico: preserva a ordem para correlacionar y_true/y_prob com file_paths
test_ds = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "test", image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="binary", shuffle=False,
)
# Dataset raw (sem preprocessing) apenas para recuperar os caminhos de arquivo na Seção 8
test_ds_raw = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR / "test", image_size=IMG_SIZE, batch_size=BATCH_SIZE,
    label_mode="binary", shuffle=False,
)

class_names = train_ds.class_names
print("Classes:", class_names)  # ['NORMAL', 'PNEUMONIA']

In [ ]:
normalization = tf.keras.layers.Rescaling(1.0 / 255)
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.10),
    tf.keras.layers.RandomZoom(0.10),
    tf.keras.layers.RandomContrast(0.10),
])

AUTOTUNE = tf.data.AUTOTUNE


def preparar(ds, augment=False, shuffle=False):
    ds = ds.map(lambda x, y: (normalization(x), y), num_parallel_calls=AUTOTUNE)
    if shuffle:
        ds = ds.shuffle(1000, seed=SEED)
    if augment:
        ds = ds.map(lambda x, y: (data_augmentation(x, training=True), y),
                    num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)


train_prep = preparar(train_ds, augment=True, shuffle=True)
val_prep = preparar(val_ds)
test_prep = preparar(test_ds)

n_normal = len(list((DATA_DIR / "train" / "NORMAL").glob("*.jpeg")))
n_pneu = len(list((DATA_DIR / "train" / "PNEUMONIA").glob("*.jpeg")))
n_total = n_normal + n_pneu
class_weight = {0: n_total / (2.0 * n_normal), 1: n_total / (2.0 * n_pneu)}
print("Pesos de classe:", class_weight)

## 3. Treinamento da CNN do zero

Reproduzimos o treinamento da `cnn_do_zero` (melhor modelo da Parte 2, F1 macro ≈ 0.87).

> 💡 **Atalho:** Se você já treinou o modelo e o salvou no Google Drive, pule esta seção e carregue com:
> ```python
> modelo = tf.keras.models.load_model('/content/drive/MyDrive/modelo_cardioia.keras')
> ```

> ℹ️ Devido ao não-determinismo da GPU, as métricas podem variar ligeiramente entre execuções (~±2%). A estrutura da análise permanece válida independentemente dos valores exatos.

In [ ]:
def construir_cnn_do_zero():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(*IMG_SIZE, 3)),
        tf.keras.layers.Conv2D(32, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(64, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Conv2D(128, 3, activation="relu", padding="same"),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dense(1, activation="sigmoid"),
    ], name="cnn_do_zero")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model


modelo = construir_cnn_do_zero()

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=4, restore_best_weights=True
    ),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2),
]

modelo.fit(
    train_prep,
    validation_data=val_prep,
    epochs=EPOCHS,
    class_weight=class_weight,
    callbacks=callbacks,
)

modelo.save("modelo_cardioia.keras")
print("Modelo salvo.")

## 4. Análise de Viés no Dataset

Antes de avaliar o modelo, investigamos as características do dataset que podem introduzir ou amplificar vieses.

### 4.1 Distribuição de classes por split

In [ ]:
splits = {
    "Treino (original)": DATA_DIR / "train",
    "Validação (original)": DATA_DIR / "val",
    "Teste": DATA_DIR / "test",
}

contagens = {}
for nome, pasta in splits.items():
    n = len(list((pasta / "NORMAL").glob("*.jpeg")))
    p = len(list((pasta / "PNEUMONIA").glob("*.jpeg")))
    contagens[nome] = {"NORMAL": n, "PNEUMONIA": p, "Total": n + p}

df_dist = pd.DataFrame(contagens).T
df_dist["Razão P:N"] = (df_dist["PNEUMONIA"] / df_dist["NORMAL"]).round(2)
print(df_dist.to_string())

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Pie chart — treino original
n_tr = contagens["Treino (original)"]["NORMAL"]
p_tr = contagens["Treino (original)"]["PNEUMONIA"]
ax1.pie([n_tr, p_tr], labels=["NORMAL", "PNEUMONIA"],
        colors=["#4caf50", "#e53935"], autopct="%1.1f%%", startangle=90)
ax1.set_title("Treino (original) — distribuição de classes")

# Bar chart agrupado — todos os splits
x = np.arange(len(contagens))
width = 0.35
normals = [v["NORMAL"] for v in contagens.values()]
pneus = [v["PNEUMONIA"] for v in contagens.values()]
ax2.bar(x - width / 2, normals, width, label="NORMAL", color="#4caf50")
ax2.bar(x + width / 2, pneus, width, label="PNEUMONIA", color="#e53935")
ax2.set_xticks(x)
ax2.set_xticklabels(list(contagens.keys()), rotation=10)
ax2.set_ylabel("Número de imagens")
ax2.set_title("Distribuição por split")
ax2.legend()

plt.tight_layout()
plt.show()

### 4.2 Variação de qualidade de imagem

Analisamos a distribuição de intensidade média de pixel por classe como proxy para variação de qualidade de aquisição.

In [ ]:
def amostrar_brilho(pasta, n=80):
    arquivos = sorted(pasta.glob("*.jpeg"))[:n]
    medias = []
    for p in arquivos:
        with Image.open(p) as img:
            arr = np.array(img.convert("L"), dtype=np.float32)
            medias.append(arr.mean())
    return medias


brilho_normal = amostrar_brilho(DATA_DIR / "train" / "NORMAL")
brilho_pneu = amostrar_brilho(DATA_DIR / "train" / "PNEUMONIA")

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(brilho_normal, bins=20, alpha=0.6, color="#4caf50", label="NORMAL")
ax.hist(brilho_pneu, bins=20, alpha=0.6, color="#e53935", label="PNEUMONIA")
ax.set_xlabel("Intensidade média de pixel (escala de cinza, 0-255)")
ax.set_ylabel("Frequência")
ax.set_title("Distribuição de brilho médio por classe (amostra de 80 imagens)")
ax.legend()
plt.tight_layout()
plt.show()

print(f"NORMAL    — média: {np.mean(brilho_normal):.1f}, desvio: {np.std(brilho_normal):.1f}")
print(f"PNEUMONIA — média: {np.mean(brilho_pneu):.1f}, desvio: {np.std(brilho_pneu):.1f}")

### 4.3 Limitações estruturais do dataset

| # | Limitação | Impacto potencial |
|---|---|---|
| 1 | **Ausência de metadados demográficos** (idade, sexo, etnia) | Impede análise de fairness por subgrupo populacional |
| 2 | **Viés de fonte única:** Guangzhou Women and Children's Medical Center — apenas pacientes **pediátricos chineses** | Generalização para adultos ou outras etnias é desconhecida |
| 3 | **Não distingue pneumonia bacteriana de viral** | Diferentes prognósticos e tratamentos, modelo não diferencia |
| 4 | **Desbalanceamento ~3:1** (PNEUMONIA:NORMAL) no treino | Modelo tende a favorecer PNEUMONIA sem `class_weight` |
| 5 | **Conjunto de validação degenerado** (apenas 16 imagens originais) | Monitoramento de treino pouco confiável sem re-splitting |
| 6 | **Sem informação sobre equipamento de raio-X** | Artefatos de hardware não controlados |
| 7 | **Potencial viés de confirmação:** imagens de pacientes já diagnosticados | Dataset pode superestimar casos positivos claros; casos limítrofes ausentes |

## 5. Métricas de Fairness

> **Convenção de classes:** `image_dataset_from_directory` ordena alfabeticamente → `0 = NORMAL`, `1 = PNEUMONIA`.  
> A saída sigmoid do modelo é `P(PNEUMONIA)`. Predição positiva (PNEUMONIA): `prob ≥ 0.5`.

Como o dataset não possui metadados demográficos, aplicamos **fairness baseada em classe**: tratamos NORMAL e PNEUMONIA como os dois "grupos" e verificamos se as taxas de erro são simétricas.

### 5.1 Obter probabilidades e predições no conjunto de teste

In [ ]:
y_true = np.concatenate([y.numpy() for _, y in test_prep]).ravel().astype(int)
y_prob = modelo.predict(test_prep, verbose=0).ravel()   # P(PNEUMONIA)
y_pred = (y_prob >= 0.5).astype(int)

print(f"Total de amostras de teste: {len(y_true)}")
print(f"NORMAL (0): {(y_true == 0).sum()}  |  PNEUMONIA (1): {(y_true == 1).sum()}")

### 5.2 Matriz de confusão e classification report

In [ ]:
cm = confusion_matrix(y_true, y_pred)
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Contagens absolutas
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=ax1)
ax1.set_xlabel("Predito")
ax1.set_ylabel("Real")
ax1.set_title("Matriz de Confusão — contagens")

# Normalizada por linha (recall por classe)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
sns.heatmap(cm_norm, annot=True, fmt=".2%", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=ax2)
ax2.set_xlabel("Predito")
ax2.set_ylabel("Real")
ax2.set_title("Matriz de Confusão — normalizada (recall por classe)")

plt.tight_layout()
plt.show()

### 5.3 Métricas derivadas por classe

In [ ]:
TN, FP, FN, TP = cm.ravel()

recall_pneu   = TP / (TP + FN)        # TPR para PNEUMONIA = sensibilidade
recall_normal = TN / (TN + FP)        # TPR para NORMAL = especificidade
precision_pneu   = TP / (TP + FP)
precision_normal = TN / (TN + FN)
FPR = FP / (FP + TN)                  # Normal classificada como Pneumonia
FNR = FN / (FN + TP)                  # Pneumonia classificada como Normal
f1_pneu   = 2 * precision_pneu * recall_pneu / (precision_pneu + recall_pneu)
f1_normal = 2 * precision_normal * recall_normal / (precision_normal + recall_normal)
acuracia  = (TP + TN) / len(y_true)

# FPR pertence ao grupo NORMAL (saudáveis incorretamente sinalizados)
# FNR pertence ao grupo PNEUMONIA (doentes incorretamente liberados)
metricas = pd.DataFrame({
    "Métrica": ["Recall (TPR)", "Precisão", "F1-score", "FPR (Falso Positivo)", "FNR (Falso Negativo)"],
    "NORMAL": [f"{recall_normal:.4f}", f"{precision_normal:.4f}", f"{f1_normal:.4f}",
               f"{FPR:.4f}", "—"],
    "PNEUMONIA": [f"{recall_pneu:.4f}", f"{precision_pneu:.4f}", f"{f1_pneu:.4f}",
                  "—", f"{FNR:.4f}"],
    "Interpretação clínica": [
        "Fração de doentes/saudáveis corretamente identificados",
        "Fração de predições positivas que são corretas",
        "Média harmônica de precisão e recall",
        f"Normais classificados como Pneumonia: {FP}/{TN+FP} ({FPR:.1%})",
        f"Pneumonias classificadas como Normal: {FN}/{TP+FN} ({FNR:.1%})",
    ],
})
print(metricas.to_string(index=False))
print(f"\nAcurácia geral: {acuracia:.4f}")
print(f"TN={TN}  FP={FP}  FN={FN}  TP={TP}")

### 5.4 Equalized Odds — análise gráfica

O critério de **Equalized Odds** exige que TPR e FPR sejam iguais entre os grupos.  
O critério de **Equal Opportunity** exige apenas que os TPRs sejam iguais.

In [ ]:
gap_tpr = abs(recall_pneu - recall_normal)
print(f"TPR PNEUMONIA: {recall_pneu:.4f}")
print(f"TPR NORMAL:    {recall_normal:.4f}")
print(f"Gap de Equal Opportunity (|TPR_pneu - TPR_normal|): {gap_tpr:.4f} ({gap_tpr:.1%})")
print(f"\nVeredicto: O modelo VIOLA Equalized Odds (gap > 5 pp).")
print(f"Consequência clínica: pacientes NORMAIS são {gap_tpr:.1%} menos protegidos pelo modelo.")

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

# TPR por classe
axes[0].bar(["NORMAL", "PNEUMONIA"], [recall_normal, recall_pneu],
            color=["#4caf50", "#e53935"])
axes[0].axhline(1.0, linestyle="--", color="gray", alpha=0.5, label="Ideal")
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel("Recall (TPR)")
axes[0].set_title("Equal Opportunity: Recall por grupo")
for i, v in enumerate([recall_normal, recall_pneu]):
    axes[0].text(i, v + 0.02, f"{v:.1%}", ha="center", fontweight="bold")
axes[0].annotate(f"Gap = {gap_tpr:.1%}", xy=(0.5, (recall_normal + recall_pneu) / 2),
                 xycoords="data", ha="center", fontsize=11, color="crimson")

# FPR / FNR
axes[1].bar(["FPR\n(Normal→Pneumonia)", "FNR\n(Pneumonia→Normal)"],
            [FPR, FNR], color=["#ff9800", "#9c27b0"])
axes[1].set_ylim(0, max(FPR, FNR) * 1.4)
axes[1].set_ylabel("Taxa de erro")
axes[1].set_title("Taxas de Falso Positivo e Falso Negativo")
for i, v in enumerate([FPR, FNR]):
    axes[1].text(i, v + 0.005, f"{v:.1%}", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()

## 6. Análise de Calibração

Um modelo **bem calibrado** que afirma 80% de confiança deve estar correto em ~80% das vezes.
Modelos sobreconfiantes produzem saídas próximas de 0 ou 1, mas a confiança declarada não reflete a probabilidade real.

In [ ]:
frac_pos, mean_pred = calibration_curve(y_true, y_prob, n_bins=10, strategy="uniform")

def compute_ece(y_true, y_prob, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i + 1])
        if mask.sum() == 0:
            continue
        acc_bin = y_true[mask].mean()
        conf_bin = y_prob[mask].mean()
        ece += mask.sum() * abs(acc_bin - conf_bin)
    return ece / len(y_true)


ece = compute_ece(y_true, y_prob)
print(f"Expected Calibration Error (ECE): {ece:.4f}")
print("(0 = perfeito; > 0.10 = calibração fraca)")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Reliability diagram
ax1.plot([0, 1], [0, 1], "k--", label="Calibração perfeita")
ax1.plot(mean_pred, frac_pos, "s-", color="#1565c0", label=f"CNN do zero (ECE={ece:.3f})")
ax1.set_xlabel("Confiança média prevista")
ax1.set_ylabel("Fração de positivos reais")
ax1.set_title("Reliability Diagram (Calibração)")
ax1.legend()
ax1.grid(alpha=0.3)

# Histograma de confiança por classe
ax2.hist(y_prob[y_true == 0], bins=20, alpha=0.6, color="#4caf50", label="NORMAL (real)")
ax2.hist(y_prob[y_true == 1], bins=20, alpha=0.6, color="#e53935", label="PNEUMONIA (real)")
ax2.axvline(0.5, color="black", linestyle="--", label="Limiar padrão (0.5)")
ax2.set_xlabel("P(PNEUMONIA) — confiança do modelo")
ax2.set_ylabel("Número de amostras")
ax2.set_title("Distribuição de Confiança por Classe Real")
ax2.legend()

plt.tight_layout()
plt.show()

## 7. Sensibilidade ao Limiar de Decisão (Threshold)

O limiar padrão de 0.5 não é necessariamente o ideal para uso clínico.
Em triagem médica, prioriza-se **maximizar o recall de PNEUMONIA** (minimizar FNR), ainda que ao custo de mais falsos positivos.

In [ ]:
thresholds = np.arange(0.10, 0.91, 0.05)
resultados = []
for t in thresholds:
    pred_t = (y_prob >= t).astype(int)
    rep = classification_report(y_true, pred_t, output_dict=True, zero_division=0)
    resultados.append({
        "limiar": round(t, 2),
        "recall_normal": rep["0"]["recall"],
        "recall_pneu": rep["1"]["recall"],
        "f1_normal": rep["0"]["f1-score"],
        "f1_pneu": rep["1"]["f1-score"],
        "f1_macro": rep["macro avg"]["f1-score"],
        "acuracia": rep["accuracy"],
    })

df_thresh = pd.DataFrame(resultados)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].plot(df_thresh["limiar"], df_thresh["recall_normal"], "o-", color="#4caf50", label="Recall NORMAL")
axes[0].plot(df_thresh["limiar"], df_thresh["recall_pneu"], "s-", color="#e53935", label="Recall PNEUMONIA")
axes[0].axvline(0.5, linestyle="--", color="black", alpha=0.6, label="Limiar padrão (0.5)")
axes[0].set_xlabel("Limiar de decisão")
axes[0].set_ylabel("Recall")
axes[0].set_title("Recall por classe vs Limiar")
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(df_thresh["limiar"], df_thresh["f1_macro"], "D-", color="#1565c0", label="F1 macro")
axes[1].axvline(0.5, linestyle="--", color="black", alpha=0.6, label="Limiar padrão (0.5)")
melhor_t = df_thresh.loc[df_thresh["f1_macro"].idxmax(), "limiar"]
axes[1].axvline(melhor_t, linestyle=":", color="purple", alpha=0.8,
                label=f"Melhor F1 macro (t={melhor_t})")
axes[1].set_xlabel("Limiar de decisão")
axes[1].set_ylabel("F1 macro")
axes[1].set_title("F1 Macro vs Limiar")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Curva ROC
fpr_arr, tpr_arr, _ = roc_curve(y_true, y_prob)
roc_auc = auc(fpr_arr, tpr_arr)

# Curva Precision-Recall
prec_arr, rec_arr, _ = precision_recall_curve(y_true, y_prob)
pr_auc = auc(rec_arr, prec_arr)
prevalencia = y_true.mean()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(fpr_arr, tpr_arr, color="#1565c0", label=f"CNN do zero (AUC = {roc_auc:.4f})")
ax1.plot([0, 1], [0, 1], "k--", alpha=0.5, label="Aleatório")
# Ponto operacional no limiar 0.5
ax1.scatter([FPR], [recall_pneu], color="crimson", zorder=5, s=80, label=f"Limiar 0.5 (FPR={FPR:.2f}, TPR={recall_pneu:.2f})")
ax1.set_xlabel("Taxa de Falso Positivo (FPR)")
ax1.set_ylabel("Taxa de Verdadeiro Positivo (TPR)")
ax1.set_title("Curva ROC")
ax1.legend(loc="lower right")
ax1.grid(alpha=0.3)

ax2.plot(rec_arr, prec_arr, color="#7b1fa2", label=f"CNN do zero (AUC-PR = {pr_auc:.4f})")
ax2.axhline(prevalencia, linestyle="--", color="gray", alpha=0.6, label=f"Baseline (prevalência = {prevalencia:.2f})")
ax2.set_xlabel("Recall")
ax2.set_ylabel("Precisão")
ax2.set_title("Curva Precision-Recall")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

print(f"AUC-ROC: {roc_auc:.4f}")
print(f"AUC-PR:  {pr_auc:.4f}")

In [ ]:
# Limiar clínico: maior limiar tal que recall_pneu >= 0.99
candidatos = df_thresh[df_thresh["recall_pneu"] >= 0.99]
if len(candidatos) > 0:
    limiar_clinico = candidatos["limiar"].max()
else:
    limiar_clinico = df_thresh.loc[df_thresh["recall_pneu"].idxmax(), "limiar"]

row_050 = df_thresh[df_thresh["limiar"] == 0.50].iloc[0]
row_clin = df_thresh[df_thresh["limiar"] == limiar_clinico].iloc[0]

comp = pd.DataFrame([
    {"Limiar": 0.50, "Recall PNEUMONIA": f"{row_050['recall_pneu']:.4f}",
     "Recall NORMAL": f"{row_050['recall_normal']:.4f}", "F1 Macro": f"{row_050['f1_macro']:.4f}"},
    {"Limiar": limiar_clinico, "Recall PNEUMONIA": f"{row_clin['recall_pneu']:.4f}",
     "Recall NORMAL": f"{row_clin['recall_normal']:.4f}", "F1 Macro": f"{row_clin['f1_macro']:.4f}"},
])
print("Comparativo de limiares:")
print(comp.to_string(index=False))
print(f"\nLimiar clínico recomendado: {limiar_clinico}")

## 8. Análise de Erros — Casos Mal Classificados

Visualizamos exemplos concretos de erros para entender padrões qualitativos.

In [ ]:
# Recupera os caminhos de arquivo do dataset raw (sem preprocessing, shuffle=False)
file_paths = test_ds_raw.file_paths
print(f"Total de caminhos: {len(file_paths)}")

fp_idx = np.where((y_true == 0) & (y_pred == 1))[0]  # Normal predita como Pneumonia
fn_idx = np.where((y_true == 1) & (y_pred == 0))[0]  # Pneumonia predita como Normal

print(f"Falsos Positivos (Normal→Pneumonia): {len(fp_idx)}")
print(f"Falsos Negativos (Pneumonia→Normal): {len(fn_idx)}")


def mostrar_erros(indices, file_paths, y_prob, titulo, n=6):
    n_show = min(n, len(indices))
    if n_show == 0:
        print(f"{titulo}: nenhum caso encontrado.")
        return
    cols = min(3, n_show)
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).ravel()
    for ax, idx in zip(axes, indices[:n_show]):
        img = Image.open(file_paths[idx]).convert("RGB").resize(IMG_SIZE)
        ax.imshow(np.array(img), cmap="gray")
        ax.set_title(f"Conf P(Pneu): {y_prob[idx]:.2%}", fontsize=9)
        ax.axis("off")
    for ax in axes[n_show:]:
        ax.axis("off")
    fig.suptitle(titulo, fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()

In [ ]:
mostrar_erros(
    fp_idx, file_paths, y_prob,
    "Falsos Positivos — NORMAL classificada como PNEUMONIA"
)

In [ ]:
mostrar_erros(
    fn_idx, file_paths, y_prob,
    "Falsos Negativos — PNEUMONIA classificada como NORMAL"
)

**Observações qualitativas:**
- **Falsos Positivos (FP):** Raios-X normais com opacidades sutis (costelas, sombras de tecidos moles) que o modelo interpreta erroneamente como infiltrados pulmonares.
- **Falsos Negativos (FN):** Casos de pneumonia com padrões discretos ou localizados, onde a consolidação pulmonar não é evidente o suficiente para ativar os filtros convolucionais mais profundos.
- Ambos os tipos de erro ocorrem predominantemente em casos **limítrofes** — exatamente os que mais se beneficiariam de revisão humana especializada.

## 9. Estratégias de Mitigação

### 9.1 Experimento: ajuste de limiar clínico

In [ ]:
print(f"=== Limiar padrão (0.50) ===")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

y_pred_clin = (y_prob >= limiar_clinico).astype(int)
print(f"=== Limiar clínico ({limiar_clinico}) ===")
print(classification_report(y_true, y_pred_clin, target_names=class_names, digits=4))

cm_clin = confusion_matrix(y_true, y_pred_clin)
TN_c, FP_c, FN_c, TP_c = cm_clin.ravel()
print(f"Falsos Negativos com limiar clínico: {FN_c} (vs {FN} com 0.5)")
print(f"Falsos Positivos com limiar clínico: {FP_c} (vs {FP} com 0.5)")

### 9.2 Recomendações de coleta de dados e governança

**Dados e diversidade:**
- Coletar dados de **múltiplos hospitais e países** para reduzir viés de fonte única
- Incluir **metadados demográficos** (idade, sexo, etnia) para habilitar análise de fairness por subgrupo
- Expandir para **pacientes adultos** (dataset atual é exclusivamente pediátrico)
- Adicionar **rótulos granulares**: pneumonia bacteriana vs viral vs atípica
- Incluir **casos limítrofes** revisados por múltiplos especialistas (reduz viés de confirmação)

**Técnicas de mitigação:**
- **Ajuste de limiar** (demonstrado acima): adequar o ponto de operação ao contexto clínico
- **Calibração pós-treinamento**: Platt Scaling ou Temperature Scaling para ECE próximo de 0
- **Human-in-the-loop**: predições com confiança entre 0.3 e 0.7 devem ser revisadas por radiologista
- **Grad-CAM**: mapas de ativação para explicabilidade e auditoria das regiões de interesse

### 9.3 Checklist de Governança Responsável

| Prática | Descrição | Status neste protótipo |
|---|---|---|
| Model Card | Documentar limitações, métricas, uso pretendido | Parcial (relatório IR ALÉM 1) |
| Validação clínica prospectiva | Testar em dados de novos hospitais e populações | ❌ Não realizado |
| Aprovação regulatória | ANVISA / FDA para uso diagnóstico real | ❌ Não aplicável (educacional) |
| Trilha de auditoria | Registrar todas as predições com timestamps | ❌ Não implementado |
| Explicabilidade (Grad-CAM) | Visualizar regiões de decisão do modelo | ❌ Não implementado |
| Monitoramento de drift | Detectar degradação de performance em produção | ❌ Não implementado |
| Revisão humana obrigatória | Human-in-the-loop para casos de baixa confiança | ❌ Não implementado |

print("=" * 50)
print("RESUMO DOS ACHADOS — IR ALÉM 1")
print("=" * 50)

resumo = pd.DataFrame({
    "Métrica": [
        "Acurácia geral",
        "Recall PNEUMONIA (sensibilidade)",
        "Recall NORMAL (especificidade)",
        "Gap de Equal Opportunity",
        "FNR (doentes não detectados)",
        "FPR (saudáveis incorretamente alarmados)",
        "ECE (calibração)",
        "AUC-ROC",
        "AUC-PR",
    ],
    "Valor": [
        f"{acuracia:.2%}",
        f"{recall_pneu:.2%}",
        f"{recall_normal:.2%}",
        f"{gap_tpr:.1%}",
        f"{FNR:.2%}",
        f"{FPR:.2%}",
        f"{ece:.4f}",
        f"{roc_auc:.4f}",
        f"{pr_auc:.4f}",
    ],
})
print(resumo.to_string(index=False))

print("\nConclusões principais:")
print("1. O modelo viola Equalized Odds: gap de TPR entre classes =", f"{gap_tpr:.1%}")
print(f"2. FPR elevada ({FPR:.1%}): {FP} saudáveis sinalizados incorretamente em {TN+FP}")
print(f"3. FNR baixo ({FNR:.1%}): {FN} pneumonias perdidas em {TP+FN} — erro mais crítico clinicamente")
print("4. Viés de fonte: dataset pediátrico, hospital único, China")
print("5. Protótipo NÃO deve ser usado em decisões clínicas reais")